In [84]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import shutil
import random

In [6]:
df = pd.read_csv("data/brand_info.csv", index_col=0) # We don't need the unnamed column
df.head()

,ID,GenderType,Type,SubType,Article,PrimaryColor,Seasonal,Year,Use,Brand
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma
8,29114,Men,Accessories,Socks,Socks,Navy Blue,Summer,2012.0,Casual,Puma
10,9204,Men,Footwear,Shoes,Casual Shoes,Black,Summer,2011.0,Casual,Puma


In [7]:
df.shape

(15137, 10)

# **MODEL 1:** `Data Preprocessing` 

## Dataset Examinations:

> We have a folder with 44,327 RGB images, each image of 60x80(Wxh). The ID column represents the name of the image file. This means we do not have to annotate the labels for each image and can simply use these rows as the annotations for each image. 

> There are only 15137 rows in the csv file which means we only have labels for these files. Rest of the image files are unlabelled. This explains that only 34.14% of the images are labeled. According to the project guidelines, we will consider rest images as "fake" and ignore them since these images have been uploaded with incorrect brand info by the 3rd party sellers.

> `This observation implies that the 34.14% images data are in the csv file which can be now tagged as "Genuine" and rest of the images data can be tagged as "Fake". This will make our class unbalanced but it is (65% to 35%) which is not huge difference and we can say this will not affect or introduce bias in the results.`

> We will build a CNN model to classify the real vs fake brands first. We will use opencv with pytorch framework for this task. When comparing two popular frameworks tensorflow and pytorch, the training time was significantly faster in pytorch while the memory usages were also high according to [this](https://arc.net/l/quote/ujakprto) article. The accuracy for both the frameworks were similar. Pytorch is more oriented towards python and is easy to interpret. So, we will use pytorch because of less training time and ease of use.

In [8]:
df.isnull().sum()

ID              0
GenderType      0
Type            0
SubType         0
Article         0
PrimaryColor    6
Seasonal        1
Year            1
Use             4
Brand           0
dtype: int64

> Since there are very less null values, we can drop them.

In [9]:
df.dropna(inplace=True)
(df.isna().sum(), df.shape)

(ID              0
 GenderType      0
 Type            0
 SubType         0
 Article         0
 PrimaryColor    0
 Seasonal        0
 Year            0
 Use             0
 Brand           0
 dtype: int64,
 (15126, 10))

In [67]:
df["ID"].duplicated().sum()

0

### Initial thought process:

- We have images in RGB, we can to convert it into grayscale, which will reduce the complexity. But the colors could be the features that determine the brand authenticity. For the purpose of this project, we will keep the images in RGB format because we might lose features if we convert them into grayscale.
- We need to create a labelled dataset for the image files which will contain the image files with their authenticity("Genuine", "Not Genuine"). We will load the files and then map the file names with the ID from the csv data. 
- We need to process the images to pass them through the CNN. We will convert them into 64x64 images now for simplicity. Based on the performance of model, we can increase this later.

### Since, pytorch provides option to resize the images, we will not resize using opencv right now.

### Create a labelled dataset for first CNN, annotating the images present in the "ID" column of csv file as "Genuine" and other image files as "Fake". 
### Now to do this, we can manually create a image-label pair and dump them in a csv file. However, pytorch happens to provide a powerful class to automatically label these images. We will be using the [ImageFolder](https://arc.net/l/quote/lpynlrki) class from pytorch to annotate the images. Steps to perform annotations will be:
- Identify the images names for "Genuine" class and put them in an array.
- Move all of the images having names that match the names in the create array in a folder "Genuine".
- Move rest of the images in the folder named "Fake".

In [124]:
genuine_ids_list = df["ID"].astype(str).to_list() # * According to the project guideline, all the IDs in csv files might not have corresponding images in the file.
len(genuine_ids_list)

15126

In [125]:
main_dir = "data/images"
genuine_dir = "data/new/genuine" # * Move to new folder
fake_dir = "data/new/fake"

os.makedirs(genuine_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)


In [126]:
fake_ids = list() # * Will be used in train test split
genuine_ids = list() 
for filename in os.listdir(main_dir):
    splitted = filename.split(".")
    file_id = splitted[0]
    ext = splitted[1]

    # * Ensure the validity of images extension
    if ext not in ["jpg"]:
        print("Not valid image...")

    if file_id in genuine_ids_list:
        shutil.move(os.path.join(main_dir, filename), os.path.join(genuine_dir, filename))
        genuine_ids_list.remove(file_id) # * According to the project guideline, all the IDs in csv files might not have corresponding images in the file. Remaining items in the genuine_ids_list will be those IDs.
        genuine_ids.append(file_id)
    else:
        fake_ids.append(file_id)
        shutil.move(os.path.join(main_dir, filename), os.path.join(fake_dir, filename))

### These IDs below does not have corresponding Images in the images folder

In [127]:
genuine_ids_list

['1164',
 '1163',
 '31244',
 '31415',
 '5026',
 '31236',
 '31252',
 '31241',
 '31284',
 '31225',
 '12347']

In [128]:
genuine_size = len(os.listdir(genuine_dir))
fake_size = len(os.listdir(fake_dir))
print(f"Genuine images: {genuine_size}")
print(f"Fake images: {fake_size}")

Genuine images: 15115
Fake images: 29211


### So, the images are classified under Genuine and Fake, now we need to split them into train, test and validate dataset. We will create these subsets for each class. We will perform (80-18-2)% (train-test-validation) split. We will have 2% of totally unseen images by the model during the process of training and testing.

> We will have a train, test and validation folder inside data folder now

In [129]:

# * Creating directories for genuine class
genuine_train_dir = "data/train/genuine"
genuine_test_dir = "data/test/genuine"
genuine_val_dir = "data/val/genuine"

os.makedirs(genuine_train_dir, exist_ok=True)
os.makedirs(genuine_test_dir, exist_ok=True)
os.makedirs(genuine_val_dir, exist_ok=True)

# * Same for fake class
fake_train_dir = "data/train/fake"
fake_test_dir = "data/test/fake"
fake_val_dir = "data/val/fake"

os.makedirs(fake_train_dir, exist_ok=True)
os.makedirs(fake_test_dir, exist_ok=True)
os.makedirs(fake_val_dir, exist_ok=True)

In [130]:
print(len(genuine_ids), len(fake_ids))
genuine_size, fake_size

15115 29211


(15115, 29211)

> Genuine id list size is same as actual genuine folder file size. This means that each of our data has corresponding image file.

In [131]:
def get_ttv_size(total_size): # * (80-18-2) split
    return (int((80/100)*total_size), int((18/100)*total_size), int((2/100)*total_size))

# * Lets shuffle the list randomly to choose random images in train, test and val sets.
random.seed(42) # * This will be similar to stratified train_test_split with seed 42
random.shuffle(genuine_ids)
random.shuffle(fake_ids)

# * Dealing with genuine data split (genuine_dir)
genuine_train_size, genuine_test_size, genuine_val_size = get_ttv_size(genuine_size)

train_genuine_ids = genuine_ids[:genuine_train_size]
test_genuine_ids = genuine_ids[genuine_train_size:genuine_train_size+genuine_test_size]
val_genuine_ids = genuine_ids[genuine_train_size+genuine_test_size:]
    
print(f"Genuine Splits: {len(train_genuine_ids), len(test_genuine_ids), len(val_genuine_ids)}")

Genuine Splits: (12092, 2720, 303)


In [132]:
# * Dealing with fake data split (fake_dir)
fake_train_size, fake_test_size, fake_val_size = get_ttv_size(fake_size)

train_fake_ids = fake_ids[:fake_train_size]
test_fake_ids = fake_ids[fake_train_size:fake_train_size+fake_test_size]
val_fake_ids = fake_ids[fake_train_size+fake_test_size:]
    
print(f"Fake Splits: {len(train_fake_ids), len(test_fake_ids), len(val_fake_ids)}")

Fake Splits: (23368, 5257, 586)


In [133]:
root_directory = 'data'
current_directory = 'data/new' # * will be moving from current dir to root dir (train, test, val)
classes = ['genuine', 'fake']
splits = ['train', 'test', 'val']

ids_mapping = {
    'train': {'genuine': train_genuine_ids, 'fake': train_fake_ids},
    'test': {'genuine': test_genuine_ids, 'fake': test_fake_ids},
    'val': {'genuine': val_genuine_ids, 'fake': val_fake_ids},
}

for stage in splits: 
    for class_name in classes: 
        source_folder = os.path.join(current_directory, class_name)
        destination_folder = os.path.join(root_directory, stage, class_name)
        
        os.makedirs(destination_folder, exist_ok=True)
        
        for image_id in ids_mapping[stage][class_name]: # * Since it is dict inside dict, we have two levels
            filename = image_id + '.jpg' # * We have ensured that all of our image files are  of .jpg format
            source_file = os.path.join(source_folder, filename)
            destination_file = os.path.join(destination_folder, filename)
            
            shutil.move(source_file, destination_file)


print((len(os.listdir(genuine_train_dir))), (len(os.listdir(genuine_test_dir))), (len(os.listdir(genuine_val_dir))))
print((len(os.listdir(fake_train_dir))), (len(os.listdir(fake_test_dir))), (len(os.listdir(fake_val_dir))))

12092 2720 303
23368 5257 586


### Nice, these numbers(folder sizes for each folder) match our previous list size. This means we have successfully moved respected percentage of files into train, test and val folders. XD